# Covertype + TabFM + XAI

This notebook runs TabFM on **CPU**, including on computers without an NVIDIA GPU. Use a standard CPU runtime in Colab. Python 3.11 or newer is required.

The pretrained weights are downloaded on the first run; the full controlled benchmark can take longer on CPU.


## 1. Install

In [ ]:
from pathlib import Path
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("TabFM requires Python 3.11 or newer. Select a compatible notebook kernel.")

# Reuse an existing clone and install the PyTorch backend into this notebook's kernel.
if not Path("tabfm").is_dir():
    !git clone -q --depth 1 https://github.com/google-research/tabfm.git
%pip install -q "./tabfm[pytorch]" lime scikit-learn scipy safetensors


## 2. The dataset
Each row is a 30m × 30m patch of forest in Colorado.
**Objective:** predict which of 7 tree types grows there (classification), using map data such as elevation, slope and distance to water.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import sys
from pathlib import Path

for _directory in (Path.cwd(), Path.cwd() / "experiments", Path.cwd() / "XAI" / "experiments"):
    if (_directory / "controlled_lime_protocol.py").is_file():
        sys.path.insert(0, str(_directory))
        break
else:
    raise FileNotFoundError("Keep controlled_lime_protocol.py beside the experiment notebooks.")

from controlled_lime_protocol import (
    CLASS_NAMES, RANDOM_STATE, SAMPLE_SIZE, TEST_SIZE, BACKGROUND_SIZE,
    SAMPLE_BUDGETS, REPEATS, EXPLANATIONS_PER_CLASS, NUM_FEATURES, TOP_K,
    FEATURE_COUNTS, load_controlled_covtype, selected_explanation_cases,
    benchmark_lime, evaluate_predictive_performance,
)

X, y, X_train, X_test, y_train, y_test = load_controlled_covtype()
feature_names = X.columns.tolist()
class_names = list(CLASS_NAMES)
lime_background = X_train.sample(n=min(BACKGROUND_SIZE, len(X_train)), random_state=RANDOM_STATE)
explanation_cases = selected_explanation_cases(X_test, y_test)

print(f"Shared modeling sample: {len(X):,} rows, {X.shape[1]} features")
print(f"Train/test: {len(X_train):,} / {len(X_test):,} rows; seed: {RANDOM_STATE}")
print(f"LIME background: {len(lime_background):,} rows")
print(f"Cases: {len(explanation_cases)} ({EXPLANATIONS_PER_CLASS} per true class)")
print("Case indices:", explanation_cases.index.tolist())
print("Perturbation dimensions:", FEATURE_COUNTS)
print("Budgets:", SAMPLE_BUDGETS, "Repeats:", REPEATS, "Explanation features:", NUM_FEATURES)
X.head()


## 3. Fit TabFM on the shared training split

Use all 8,000 protocol training rows and the same 2,000 held-out rows as the other notebooks. TabFM uses a pretrained foundation model and the training rows as context; its fit does not train the foundation model from scratch. Keep `controlled_lime_protocol.py` beside this notebook, including in Colab.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Shared training/test shapes:", X_train.shape, X_test.shape)


In [ ]:
import torch
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1

torch.manual_seed(RANDOM_STATE)

# Explicit CPU execution keeps this experiment portable across machines.
tabfm_model = tabfm_v1.load(device="cpu", dtype=torch.float32)
print("TabFM compute device:", next(tabfm_model.parameters()).device)
model = TabFMClassifier(
    model=tabfm_model, n_estimators=8, cache_context=True,
    random_state=RANDOM_STATE, max_num_rows=None,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, labels=list(range(7)), target_names=class_names, zero_division=0))

## 4. XAI 1: Permutation feature importance (global)
Shuffle one feature at a time and see how much accuracy drops. A bigger drop means the model relies on that feature more.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

result = permutation_importance(model, X_test[:500], y_test[:500], n_repeats=3, random_state=RANDOM_STATE)
importance = pd.Series(result.importances_mean, index=X.columns).sort_values()

importance.tail(10).plot.barh(title="Top 10 features (permutation importance)")
plt.xlabel("Drop in accuracy")
plt.show()

## 5. XAI 2: LIME (local)
Explain why the model made its prediction for one specific patch of forest.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# Use the protocol's background and categorical handling for the illustration too.
categorical_feature_indices = list(range(10, X.shape[1]))
explainer = LimeTabularExplainer(
    training_data=lime_background.to_numpy(),
    feature_names=feature_names,
    class_names=class_names,
    categorical_features=categorical_feature_indices,
    categorical_names={i: ["0", "1"] for i in categorical_feature_indices},
    mode="classification",
    random_state=RANDOM_STATE,
)

def predict_proba_for_lime(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=feature_names))

# Select the same protocol case in every notebook, independently of predictions.
case_index = explanation_cases.index[0]
instance = X_test.loc[case_index]
true_class = int(y_test.loc[case_index])
predicted_class = int(model.predict(X_test.loc[[case_index]])[0])
predicted_class_index = int(np.flatnonzero(model.classes_ == predicted_class)[0])
probabilities = predict_proba_for_lime(instance.to_numpy()[None, :])[0]
print("Shared test row:", case_index)
print("Actual:", class_names[true_class], "| Predicted:", class_names[predicted_class])

# This single explanation illustrates the result; the comparison uses all shared cases.
explanation = explainer.explain_instance(
    data_row=instance.to_numpy(),
    predict_fn=predict_proba_for_lime,
    labels=(predicted_class_index,),
    num_features=NUM_FEATURES,
    num_samples=SAMPLE_BUDGETS[-2],
)

explanation.as_pyplot_figure(label=predicted_class_index)
plt.show()


## Controlled LIME comparison

`controlled_lime_protocol.py` is the source of shared experiment settings. Every model receives the same stratified 10,000-row sample, 8,000 training rows, 2,000 held-out rows, 54 input predictors, and labels 0–6. The protocol chooses three held-out cases per true class without filtering by predictions, then uses the same 2,000-row training background, budgets, five repeat seeds (42–46), ten explanation terms, and stability definitions.

The 18/36/54-feature subsets change which predictors LIME perturbs. Models are fitted once on all 54 predictors; excluded predictors remain fixed at each case's value. Runtime includes LIME's prediction calls and surrogate fitting but excludes explainer setup and prediction warm-up. Fidelity is the local surrogate's weighted R², not predictive accuracy. Stability uses signed-weight Spearman on selected-feature unions and top-five Jaccard.

Model architecture, its hyperparameters and required preprocessing remain model-specific. This notebook uses CPU. Compare runtime on the same computer and with comparable CPU resources across models. Results below must be regenerated after changing the protocol.


In [ ]:
# The shared function owns sampling, seeds, feature subsets and stability calculations.
model_name = 'TabFM'
controlled_raw, controlled_summary = benchmark_lime(
    model, X_train, X_test, y_test, predict_proba_for_lime, model_name
)
predictive_metrics = evaluate_predictive_performance(model, X_test, y_test)
print("Held-out predictive performance:", predictive_metrics)
display(controlled_summary.round(4))

# Write beside the protocol, regardless of whether Jupyter started at the repo root.
import controlled_lime_protocol as protocol
output_directory = Path(protocol.__file__).resolve().parent
controlled_raw.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_raw.csv", index=False)
controlled_summary.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_summary.csv", index=False)
pd.DataFrame([{"model": model_name, **predictive_metrics}]).to_csv(
    output_directory / f"{model_name.lower()}_controlled_predictive_metrics.csv", index=False
)


## Controlled results across budgets and perturbation dimensions


In [ ]:
# Plot the shared summary separately for each perturbation dimension.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for feature_count, group in controlled_summary.groupby("feature_count"):
    group = group.sort_values("num_samples")
    label = f"{feature_count} perturbed features"
    axes[0, 0].errorbar(group["num_samples"], group["runtime_mean_seconds"],
        yerr=group["runtime_std_seconds"], marker="o", capsize=3, label=label)
    axes[0, 1].errorbar(group["num_samples"], group["fidelity_mean"],
        yerr=group["fidelity_std"], marker="o", capsize=3, label=label)
    axes[1, 0].plot(group["num_samples"], group["spearman_stability_mean"], marker="o", label=label)
    axes[1, 1].plot(group["num_samples"], group["top5_jaccard_mean"], marker="o", label=label)

for ax, title, ylabel in zip(axes.flat,
    ["Explanation runtime", "Local fidelity", "Signed-weight rank stability", "Top-five feature stability"],
    ["Seconds (mean ± SD)", "Weighted surrogate R² (mean ± SD)", "Mean pairwise Spearman", "Mean pairwise Jaccard"]):
    ax.set(title=title, xlabel="Perturbation samples", ylabel=ylabel, xscale="log")
    ax.set_xticks(SAMPLE_BUDGETS, labels=[f"{budget:,}" for budget in SAMPLE_BUDGETS])
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0, 0].set_ylim(bottom=0)
axes[1, 0].set_ylim(-1.05, 1.05)
axes[1, 1].set_ylim(-0.05, 1.05)
fig.suptitle(f"{model_name}: controlled LIME comparison")
plt.tight_layout()
plt.show()
